In [2]:
!pip install jiwer

  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
Using cached jiwer-4.0.0-py3-none-any.whl (23 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 3.4 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 3.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [rapidfuzz]
   ---------------------------------------- 0/2 [rapidfuzz]
   -------------------- ------------------- 1/2 [jiwer]
   ---------------------------------------- 2/2 [jiwer]




[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import re
import unicodedata
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from jiwer import wer

In [18]:
# ============================================================
# Paths
# ============================================================

TRANSCRIPTION_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\audio_context_mapping_PhoASR_VIQUAD_val.json")
GROUNDTRUTH_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\uit_viquad_val.json")
OUTPUT_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\wer_calc_PhoASR_VIQUAD_val_last.json")
TOP_K = 5

In [4]:
# ============================================================
# Text normalization
# ============================================================

def normalize_text(text):
    """
    Normalize text before matching.

    Vietnamese diacritics are preserved.
    """

    text = unicodedata.normalize("NFC", text)
    text = text.lower()

    # Remove punctuation
    text = re.sub(
        r"[^\w\sÀ-ỹ]",
        " ",
        text,
        flags=re.UNICODE
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# Extract context ID prefix
# ============================================================

def get_context_id_prefix(qa_id):

    parts = qa_id.split("-")

    if len(parts) < 2:
        return None

    return f"{parts[0]}-{parts[1]}"

In [19]:
# ============================================================
# Load files
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcription_data = json.load(f)

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth_data = json.load(f)


# ============================================================
# Extract contexts and index them by ID
# ============================================================

def extract_contexts_by_id(data):

    contexts = {}

    for data_item in data["data"]:

        for paragraph in data_item["paragraphs"]:

            context = paragraph["context"]
            qas = paragraph["qas"]

            if not context or not qas:
                continue

            context_id = get_context_id_prefix(
                qas[0]["id"]
            )

            contexts[context_id] = {
                "context_id": context_id,
                "context": context,
                "qas": qas
            }

    return contexts

transcription_contexts = extract_contexts_by_id(
    transcription_data
)

groundtruth_contexts = extract_contexts_by_id(
    groundtruth_data
)


print(
    f"Loaded {len(transcription_contexts)} "
    f"contexts from transcription file"
)

print(
    f"Loaded {len(groundtruth_contexts)} "
    f"contexts from ground-truth file"
)


# ============================================================
# Storage for results
# ============================================================

results = []
rejected_files = []


# ============================================================
# Match contexts by ID
# ============================================================

for context_id, transcription_item in (
    transcription_contexts.items()
):

    # --------------------------------------------------------
    # Check whether matching ground-truth context exists
    # --------------------------------------------------------

    if context_id not in groundtruth_contexts:

        rejected_files.append({
            "context_id": context_id,
            "transcription": transcription_item["context"],
            "reason": "context_id_not_found_in_groundtruth"
        })

        continue


    groundtruth_item = groundtruth_contexts[context_id]


    # --------------------------------------------------------
    # Get contexts
    # --------------------------------------------------------

    transcription = transcription_item["context"]
    groundtruth = groundtruth_item["context"]


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    normalized_transcription = normalize_text(
        transcription
    )

    normalized_groundtruth = normalize_text(
        groundtruth
    )


    # --------------------------------------------------------
    # Handle empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_files.append({
            "context_id": context_id,
            "transcription": transcription,
            "matched_context": groundtruth,
            "reason": "empty_transcription"
        })

        continue



    # --------------------------------------------------------
    # Calculate WER
    #
    # IMPORTANT:
    # WER(reference, hypothesis)
    #
    # Ground truth = reference
    # Transcription = hypothesis
    # --------------------------------------------------------

    context_wer = wer(
        normalized_groundtruth,
        normalized_transcription
    )


        # ----------------------------------------------------
        # Store accepted result
        # ----------------------------------------------------

    results.append({
            "context_id": context_id,

            "transcription": transcription,

            "matched_context": groundtruth,

            "qas": groundtruth_item["qas"],

            "wer": float(context_wer)
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if len(results) % 10 == 0:

        print(
            f"Processed "
            f"{len(results)} contexts"
        )


# ============================================================
# Check for ground-truth contexts that have no transcription
# ============================================================

missing_transcriptions = []

for context_id in groundtruth_contexts:

    if context_id not in transcription_contexts:

        missing_transcriptions.append({
            "context_id": context_id,
            "groundtruth": (
                groundtruth_contexts[context_id]["context"]
            ),
            "reason": "context_id_not_found_in_transcription"
        })


# ============================================================
# Calculate average WER overall
# ============================================================

if results:

    average_wer = sum(
        result["wer"]
        for result in results
    ) / len(results)

else:

    average_wer = None


# ============================================================
# Save accepted matches + summary
# ============================================================

output_data = {
    "summary": {
        "transcription_contexts": len(transcription_contexts),
        "groundtruth_contexts": len(groundtruth_contexts),
        "accepted_results": len(results),
        "missing_transcriptions": len(missing_transcriptions),
        "rejected_files": len(rejected_files),
        "average_wer": average_wer,
        "average_wer_percent": (
            average_wer * 100
            if average_wer is not None
            else None
        )
    },

    "results": results
}


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        output_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Summary
# ============================================================

print()
print("============================================================")
print("Summary")
print("============================================================")

print(
    f"Transcription contexts: "
    f"{len(transcription_contexts)}"
)

print(
    f"Ground-truth contexts: "
    f"{len(groundtruth_contexts)}"
)

print(
    f"Accepted results: "
    f"{len(results)}"
)

print(
    f"Missing transcriptions: "
    f"{len(missing_transcriptions)}"
)

print(
    f"Rejected files: "
    f"{len(rejected_files)}"
)

if average_wer is not None:

    print(
        f"Average WER overall: "
        f"{average_wer:.4f} "
        f"({average_wer * 100:.2f}%)"
    )

else:

    print("Average WER overall: N/A (no accepted results)")

FileNotFoundError: [Errno 2] No such file or directory: 'G:\\Uni\\Sem2\\NLP\\term_paper\\data\\ViSQA\\ViSQA\\transcription\\audio_context_mapping_PhoASR_VIQUAD_val.json'

In [14]:
print(type(transcription_data))
print(type(transcription_data[0]))
print(transcription_data[0].keys())

print("transcription type:",
      type(transcription_data[0]["transcription"]))

print("transcription value:")
print(transcription_data[0]["transcription"])


<class 'list'>
<class 'dict'>
dict_keys(['file', 'path', 'transcription'])
transcription type: <class 'str'>
transcription value:
Ngày nay, Pue Tor Rico là một quốc gia có nền kinh tế cơ cấu hiện đại với nông nghiệp chỉ chiếm một phần trăm, công nghiệp chiếm bốn mươi lăm phần trăm và dịch vụ chiếm năm mươi bốn phần trăm. Ngành trồng mía đường này đã nhường lại cho ngành chăn nuôi lấy sản phẩm từ thịt và sữa. Công nghiệp của Pue Tor Rico phát triển đa dạng và nhấn mạnh các ngành hóa dầu, dược phẩm và khoa học công nghệ du lịch là một trong những ngành dịch vụ chủ đạo của Pue Tor Rico với việc nước này đón tiếp năm phẩy chín triệu lượt du khách vào năm hai nghìn không trăm linh bảy đem lại một nguồn doanh thu lớn cho hòn đảo. Nhiều khách sạn các khu nghỉ dưỡng, trung tâm hội thảo lớn như trung tâm hội nghị Pua El Torico được xây dựng đã tận dụng triệt để ưu thế du lịch của đất nước tươi đẹp này.


In [15]:
print(transcription_data[0])

{'file': 'ctx_006fe94a.mp3', 'path': '/content/ViSQA/ViSQA/speech_data/val/val/contexts/ctx_006fe94a.mp3', 'transcription': 'Ngày nay, Pue Tor Rico là một quốc gia có nền kinh tế cơ cấu hiện đại với nông nghiệp chỉ chiếm một phần trăm, công nghiệp chiếm bốn mươi lăm phần trăm và dịch vụ chiếm năm mươi bốn phần trăm. Ngành trồng mía đường này đã nhường lại cho ngành chăn nuôi lấy sản phẩm từ thịt và sữa. Công nghiệp của Pue Tor Rico phát triển đa dạng và nhấn mạnh các ngành hóa dầu, dược phẩm và khoa học công nghệ du lịch là một trong những ngành dịch vụ chủ đạo của Pue Tor Rico với việc nước này đón tiếp năm phẩy chín triệu lượt du khách vào năm hai nghìn không trăm linh bảy đem lại một nguồn doanh thu lớn cho hòn đảo. Nhiều khách sạn các khu nghỉ dưỡng, trung tâm hội thảo lớn như trung tâm hội nghị Pua El Torico được xây dựng đã tận dụng triệt để ưu thế du lịch của đất nước tươi đẹp này.'}


In [6]:
# ============================================================
# Storage for results
# ============================================================

results = []
rejected_files = []


# ============================================================
# Match contexts by ID
# ============================================================

for context_id, transcription_item in (
    transcription_contexts.items()
):

    # --------------------------------------------------------
    # Check whether matching ground-truth context exists
    # --------------------------------------------------------

    if context_id not in groundtruth_contexts:

        rejected_files.append({
            "context_id": context_id,
            "transcription": transcription_item["context"],
            "reason": "context_id_not_found_in_groundtruth"
        })

        continue


    groundtruth_item = groundtruth_contexts[context_id]


    # --------------------------------------------------------
    # Get contexts
    # --------------------------------------------------------

    transcription = transcription_item["context"]
    groundtruth = groundtruth_item["context"]


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    normalized_transcription = normalize_text(
        transcription
    )

    normalized_groundtruth = normalize_text(
        groundtruth
    )


    # --------------------------------------------------------
    # Handle empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_files.append({
            "context_id": context_id,
            "transcription": transcription,
            "matched_context": groundtruth,
            "reason": "empty_transcription"
        })

        continue



    # --------------------------------------------------------
    # Calculate WER
    #
    # IMPORTANT:
    # WER(reference, hypothesis)
    #
    # Ground truth = reference
    # Transcription = hypothesis
    # --------------------------------------------------------

    context_wer = wer(
        normalized_groundtruth,
        normalized_transcription
    )


        # ----------------------------------------------------
        # Store accepted result
        # ----------------------------------------------------

    results.append({
            "context_id": context_id,

            "transcription": transcription,

            "matched_context": groundtruth,

            "qas": groundtruth_item["qas"],

            "wer": float(context_wer)
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if len(results) % 10 == 0:

        print(
            f"Processed "
            f"{len(results)} contexts"
        )


# ============================================================
# Check for ground-truth contexts that have no transcription
# ============================================================

missing_transcriptions = []

for context_id in groundtruth_contexts:

    if context_id not in transcription_contexts:

        missing_transcriptions.append({
            "context_id": context_id,
            "groundtruth": (
                groundtruth_contexts[context_id]["context"]
            ),
            "reason": "context_id_not_found_in_transcription"
        })


# ============================================================
# Calculate average WER overall
# ============================================================

if results:

    average_wer = sum(
        result["wer"]
        for result in results
    ) / len(results)

else:

    average_wer = None


# ============================================================
# Save accepted matches + summary
# ============================================================

output_data = {
    "summary": {
        "transcription_contexts": len(transcription_contexts),
        "groundtruth_contexts": len(groundtruth_contexts),
        "accepted_results": len(results),
        "missing_transcriptions": len(missing_transcriptions),
        "rejected_files": len(rejected_files),
        "average_wer": average_wer,
        "average_wer_percent": (
            average_wer * 100
            if average_wer is not None
            else None
        )
    },

    "results": results
}


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        output_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Summary
# ============================================================

print()
print("============================================================")
print("Summary")
print("============================================================")

print(
    f"Transcription contexts: "
    f"{len(transcription_contexts)}"
)

print(
    f"Ground-truth contexts: "
    f"{len(groundtruth_contexts)}"
)

print(
    f"Accepted results: "
    f"{len(results)}"
)

print(
    f"Missing transcriptions: "
    f"{len(missing_transcriptions)}"
)

print(
    f"Rejected files: "
    f"{len(rejected_files)}"
)

if average_wer is not None:

    print(
        f"Average WER overall: "
        f"{average_wer:.4f} "
        f"({average_wer * 100:.2f}%)"
    )

else:

    print("Average WER overall: N/A (no accepted results)")


NameError: name 'transcription_contexts' is not defined

In [7]:
MAX_WER = 2.0
# ============================================================
# Load ASR transcriptions
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcriptions = json.load(f)


# ============================================================
# Load ViSQA JSON
# ============================================================

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth = json.load(f)


# ============================================================
# Extract ground-truth contexts
# ============================================================

contexts = []

for data_item in groundtruth["data"]:

    for paragraph in data_item["paragraphs"]:

        context = paragraph["context"]
        qas = paragraph["qas"]

        if not context or not qas:
            continue

        # Context ID comes from the first QA ID.
        #
        # Example:
        # 0057-0077-0001
        #       ↓
        # 0057-0077

        context_id = get_context_id_prefix(
            qas[0]["id"]
        )

        contexts.append({
            "context_id": context_id,
            "context": context,
            "qas": qas
        })


print(
    f"Loaded {len(contexts)} ground-truth contexts"
)

print(
    f"Loaded {len(transcriptions)} audio transcriptions"
)


# ============================================================
# Normalize ground-truth contexts
# ============================================================

normalized_contexts = [
    normalize_text(item["context"])
    for item in contexts
]


# ============================================================
# Character n-gram TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True
)


context_vectors = vectorizer.fit_transform(
    normalized_contexts
)


# ============================================================
# Storage
# ============================================================

results = []
rejected_files = []


# ============================================================
# Process every transcription
# ============================================================

for i, item in enumerate(transcriptions):

    audio_file = item.get("file", "")
    transcription = item.get("transcription", "")

    # --------------------------------------------------------
    # Normalize transcription
    # --------------------------------------------------------

    normalized_transcription = normalize_text(
        transcription
    )


    # --------------------------------------------------------
    # Handle empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_files.append({
            "audio_file": audio_file,
            "transcription": transcription,
            "reason": "empty_transcription"
        })

        continue


    # ========================================================
    # Find the most similar ground-truth contexts
    # ========================================================

    transcription_vector = vectorizer.transform(
        [normalized_transcription]
    )


    similarities = cosine_similarity(
        transcription_vector,
        context_vectors
    )[0]


    # --------------------------------------------------------
    # Retrieve top-k candidates
    # --------------------------------------------------------

    top_indices = similarities.argsort()[-TOP_K:][::-1]


    candidates = []


    # ========================================================
    # Calculate WER for each candidate
    # ========================================================

    for idx in top_indices:

        candidate_context = normalized_contexts[idx]

        candidate_wer = wer(
            candidate_context,
            normalized_transcription
        )


        candidates.append({
            "context_id": contexts[idx]["context_id"],

            "similarity": float(
                similarities[idx]
            ),

            "wer": float(candidate_wer)
        })


    # ========================================================
    # Select candidate with lowest WER
    # ========================================================

    best_candidate = min(
        candidates,
        key=lambda x: x["wer"]
    )


    best_index = next(
        idx
        for idx in top_indices
        if contexts[idx]["context_id"]
        == best_candidate["context_id"]
    )


    best_context = contexts[best_index]


    # ========================================================
    # Check WER cutoff
    # ========================================================

    if best_candidate["wer"] > MAX_WER:

        rejected_files.append({

            "audio_file": audio_file,

            "transcription": transcription,

            "matched_context": best_context["context"],

            "context_id": best_candidate["context_id"],

            "similarity": best_candidate["similarity"],

            "wer": best_candidate["wer"],

            "wer_cutoff": MAX_WER,

            "reason": "best_candidate_exceeds_wer_cutoff",

            "candidates": candidates
        })


    else:

        # ====================================================
        # Store accepted result
        # ====================================================

        results.append({

            "audio_file": audio_file,

            "transcription": transcription,

            "context_id": best_candidate["context_id"],

            "matched_context": best_context["context"],

            "qas": best_context["qas"],

            "similarity": best_candidate["similarity"],

            "wer": best_candidate["wer"],

            "wer_percent": (
                best_candidate["wer"] * 100
            ),

            "candidates": candidates
        })


    # ========================================================
    # Progress
    # ========================================================

    if (i + 1) % 10 == 0:

        print(
            f"Processed "
            f"{i + 1}/{len(transcriptions)}"
        )


# ============================================================
# Calculate average WER
# ============================================================

if results:

    average_wer = (
        sum(result["wer"] for result in results)
        / len(results)
    )

else:

    average_wer = None


# ============================================================
# Save accepted results
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        {
            "summary": {
                "total_transcriptions": len(transcriptions),

                "total_groundtruth_contexts": len(contexts),

                "accepted_results": len(results),

                "rejected_files": len(rejected_files),

                "average_wer": average_wer,

                "average_wer_percent": (
                    average_wer * 100
                    if average_wer is not None
                    else None
                )
            },

            "results": results
        },

        f,

        ensure_ascii=False,

        indent=2
    )


# ============================================================
# Save rejected files
# ============================================================

with open(REJECTED_FILE, "w", encoding="utf-8") as f:

    json.dump(
        rejected_files,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Summary
# ============================================================

print()
print("============================================================")
print("Summary")
print("============================================================")

print(
    f"Transcriptions: "
    f"{len(transcriptions)}"
)

print(
    f"Ground-truth contexts: "
    f"{len(contexts)}"
)

print(
    f"Accepted results: "
    f"{len(results)}"
)

print(
    f"Rejected files: "
    f"{len(rejected_files)}"
)

if average_wer is not None:

    print(
        f"Average WER: "
        f"{average_wer:.4f} "
        f"({average_wer * 100:.2f}%)"
    )

else:

    print(
        "Average WER: N/A "
        "(no accepted results)"
    )


Loaded 557 ground-truth contexts
Loaded 557 audio transcriptions
Processed 10/557
Processed 20/557
Processed 30/557
Processed 40/557
Processed 50/557
Processed 60/557
Processed 70/557
Processed 80/557
Processed 90/557
Processed 100/557
Processed 110/557
Processed 120/557
Processed 130/557
Processed 140/557
Processed 150/557
Processed 160/557
Processed 170/557
Processed 180/557
Processed 190/557
Processed 200/557
Processed 210/557
Processed 220/557
Processed 230/557
Processed 240/557
Processed 250/557
Processed 260/557
Processed 270/557
Processed 280/557
Processed 290/557
Processed 300/557
Processed 310/557
Processed 320/557
Processed 330/557
Processed 340/557
Processed 350/557
Processed 360/557
Processed 370/557
Processed 380/557
Processed 390/557
Processed 400/557
Processed 410/557
Processed 420/557
Processed 430/557
Processed 440/557
Processed 450/557
Processed 460/557
Processed 470/557
Processed 480/557
Processed 490/557
Processed 500/557
Processed 510/557
Processed 520/557
Processe

NameError: name 'REJECTED_FILE' is not defined

In [2]:
JSON_PATH = r"G:\Uni\Sem2\NLP\term_paper\data\base_vit5_test_results.json"

In [20]:
from pathlib import Path
import json

JSON_PATH = Path(r"G:\Uni\Sem2\NLP\term_paper\data\base_vit5_phoasr_test_results.json")

# Load JSON
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

results = data["results"]

# ============================================================
# Calculate statistics
# ============================================================

num_examples = len(results)

asr_wers = [x["asr_wer"] for x in results]
vit5_wers = [x["vit5_wer"] for x in results]

mean_asr_wer = sum(asr_wers) / num_examples
mean_vit5_wer = sum(vit5_wers) / num_examples

absolute_reduction = (
    mean_asr_wer - mean_vit5_wer
)

relative_reduction = (
    absolute_reduction / mean_asr_wer
    if mean_asr_wer > 0
    else 0
)

improved = sum(
    x["vit5_wer"] < x["asr_wer"]
    for x in results
)

unchanged = sum(
    x["vit5_wer"] == x["asr_wer"]
    for x in results
)

worsened = sum(
    x["vit5_wer"] > x["asr_wer"]
    for x in results
)


# ============================================================
# Print
# ============================================================

print("=" * 80)
print("BASE ViT5 — TEST RESULTS - PhoASR")
print("=" * 80)

print(f"Examples:               {num_examples}")
print(f"Mean ASR WER:           {mean_asr_wer:.4f}")
print(f"Mean Base ViT5 WER:     {mean_vit5_wer:.4f}")
print(f"Absolute WER reduction: {absolute_reduction:.4f}")
print(
    f"Relative WER reduction: "
    f"{relative_reduction * 100:.2f}%"
)

print()
print(f"Improved:               {improved}")
print(f"Unchanged:              {unchanged}")
print(f"Worsened:               {worsened}")

print("=" * 80)

BASE ViT5 — TEST RESULTS - PhoASR
Examples:               557
Mean ASR WER:           0.3171
Mean Base ViT5 WER:     1.9901
Absolute WER reduction: -1.6730
Relative WER reduction: -527.68%

Improved:               2
Unchanged:              0
Worsened:               555
